In [15]:
import numpy as np
import pandas as pd
from scipy import stats

In [16]:
active_validators_size = pd.read_csv('../int/active_validators_size_change.csv')
active_validators_category = pd.read_csv('../int/active_validators_category_change.csv')
active_validators_pool = pd.read_csv('../int/active_validators_pool_change.csv')

In [17]:
active_validators_pool = active_validators_pool.drop(columns=('Unnamed: 0'))
active_validators_category = active_validators_category.drop(columns=('Unnamed: 0'))
active_validators_size = active_validators_size.drop(columns=('Unnamed: 0'))

In [18]:
curve = pd.read_csv('../int/curve_grouped.csv', usecols=('slot', 'liquidity_apr'))
curve['price_pct_change'] = curve['liquidity_apr'].pct_change()
curve = curve.dropna()
curve = curve[curve['slot'].isin(active_validators_size['slot'])]
curve = curve[curve['slot'] >= 6206400]
curve

,liquidity_apr,slot,price_pct_change
466,3.238887,6206400,0.024363
467,3.088169,6213600,-0.046534
468,3.478843,6220800,0.126507
469,3.141072,6228000,-0.097093
470,2.849985,6235200,-0.092671
...,...,...,...
848,1.883774,8956800,-0.021273
849,2.065560,8964000,0.096501
850,2.160793,8971200,0.046105
851,2.136529,8978400,-0.011229


In [19]:

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(active_validators_size, curve[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total'] / price_elasticity_size['price_pct_change']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[col] / price_elasticity_size['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)

Elasticity Analysis Results:
total: Mean Elasticity = 0.06653654205804538, t(Mean) = -, SD = 2.391561739965001, N = 387, p-value = -
1: Mean Elasticity = -0.18074966857140978, t(Mean) = -0.9315450476055828, SD = 4.642365416174716, N = 387, p-value = 0.3519610235619023
2-5: Mean Elasticity = -0.28005113503266565, t(Mean) = -1.5624551963097284, SD = 3.6500477661711446, N = 387, p-value = 0.11865592241804447
6-19: Mean Elasticity = -0.012866915576185575, t(Mean) = -0.3984247527849304, SD = 3.1066472527397915, N = 387, p-value = 0.6904344868486716
20-99: Mean Elasticity = -1.3049452471650111, t(Mean) = -1.3213199577906791, SD = 20.278606371211144, N = 387, p-value = 0.18715620387266355
100+: Mean Elasticity = 0.14258274742143118, t(Mean) = 0.36735524839756095, SD = 3.296148609069992, N = 387, p-value = 0.7134644163706936
          slot         1      100+       2-5     20-99      6-19     total  \
0    6206400.0  0.015042 -0.015342  0.000000  0.018732  0.000000 -0.012792   
1    6213600.0 

In [20]:

# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(active_validators_category, curve[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total'] / price_elasticity_category['price_pct_change']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = active_validators_category.columns.drop('slot')
for col in columns:
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[col] / price_elasticity_category['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = 0.06653654205804538, t(Mean) = 0.0, SD = 2.391561739965001, N = 387, p-value = 1.0
CEX: Mean Elasticity = -0.1033863997661892, t(Mean) = -1.1495911648069868, SD = 1.6539999421909555, N = 387, p-value = 0.2507126281550018
Liquid Restaking: Mean Elasticity = 12.986660294379462, t(Mean) = 1.1684424332097272, SD = 217.51468689406843, N = 387, p-value = 0.2433493042939361
Liquid Staking: Mean Elasticity = 0.24649829728550401, t(Mean) = 0.8269631059283, SD = 3.550740738141999, N = 387, p-value = 0.4085495827953246
Solo Stakers: Mean Elasticity = -0.1044329702050738, t(Mean) = -1.260210891261359, SD = 1.1846579878782186, N = 387, p-value = 0.20811383223730692
Staking Pools: Mean Elasticity = 0.037119247562003375, t(Mean) = -0.13047988973417796, SD = 3.735178740283344, N = 387, p-value = 0.8962267068890366
Unidentified: Mean Elasticity = -0.7446592272583111, t(Mean) = -1.576654747375005, SD = 9.834889039482714, N = 387, p-value = 0.11560782

In [21]:

# Merge the two DataFrames on the slot column
price_elasticity_pool = pd.merge(active_validators_pool, curve[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['total'] / price_elasticity_pool['price_pct_change']
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[col] / price_elasticity_pool['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_pool[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = 0.06653654205804538, t(Mean) = -, SD = 2.391561739965001, N = 387, p-value = -
Lido: Mean Elasticity = -0.003542508437232124, t(Mean) = -0.4320806302293201, SD = 2.1120291357034766, N = 387, p-value = 0.6658053114266016
Coinbase: Mean Elasticity = -0.015518958712456553, t(Mean) = -0.47613684934808176, SD = 2.4029585381798664, N = 387, p-value = 0.6341116057337205
Binance: Mean Elasticity = 0.06541559698946099, t(Mean) = -0.0069785985751458805, SD = 2.065265189030517, N = 387, p-value = 0.9944337702711608
Rocketpool: Mean Elasticity = 1.1242978310098766, t(Mean) = 1.3096820431783391, SD = 15.707270020532162, N = 387, p-value = 0.19104789809033687
Kraken: Mean Elasticity = -0.2991182671458292, t(Mean) = -1.179634451538195, SD = 5.609335980991237, N = 387, p-value = 0.23868312650744863
OKX: Mean Elasticity = -0.6449396068173048, t(Mean) = -1.420234956480014, SD = 9.560387594143943, N = 387, p-value = 0.15625705076474697
Bitcoin Suisse: